# Week 8 - Advanced Visualization with Seaborn

**Paññāsāstra University of Cambodia**
**Course:** Data Analysis and Visualization (3 credits)
**Chapter 8 - Advanced Visualization with Seaborn**

---

## Session topics (from the course outline)

- Distribution plots
- Box plots, violin plots
- Pair plots, heatmaps

## Course learning outcomes addressed

| CLO | Statement |
|---|---|
| **CLO 3** | Apply Python libraries (NumPy, Pandas) to clean, manipulate and process datasets |
| **CLO 8** | Evaluate the effectiveness of different visualization methods for communicating insights |
| **CLO 9** | Design interactive dashboards or visual reports to support data-driven decisions |

## Reading

| Topic | Source |
|---|---|
| Plotting with pandas and seaborn | McKinney §9.2 |
| Histograms and density plots | McKinney §9.2 |
| Scatter or point plots | McKinney §9.2 |
| Facet grids and categorical data | McKinney §9.2 |
| Heatmaps | Pfaffelhuber §8.10 |

## What seaborn adds

seaborn is a layer over matplotlib, not a replacement for it. Everything from Week 7 still
applies: there is still a Figure, there are still Axes, and you still label them yourself.

What seaborn adds is three things:

1. **Statistical defaults.** `sns.barplot` aggregates and draws a confidence interval without
   being asked. `sns.regplot` fits a line. The statistics are done for you, which is
   convenient and occasionally dangerous, because a default you did not choose is still a
   choice you are publishing.
2. **Data-aware syntax.** You pass a DataFrame and the names of columns, and seaborn handles
   the grouping, the colours and the legend.
3. **Plots that are tedious in matplotlib.** A violin plot, a pair plot or an annotated
   heatmap is one line here and forty there.

The session's central question is a return to Week 1. Anscombe's quartet showed that four
datasets can share every summary statistic and look completely different. **A box plot is a
summary statistic drawn as a picture** and the same problem applies to it.

---

## Setup

All imports for this notebook are defined in the single cell below. Run it first. If any
import fails, return to Notebook 00 and check that the kernel in the top-right corner points
inside your `venv`.

In [ ]:
# ---------------------------------------------------------------------------
# All imports for this notebook. Run this cell first.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Reproducibility: every dataset in this notebook is generated from this seed,
# so your numbers will match the ones shown in the lecture exactly.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Consistent, colour-blind-safe plotting defaults for the whole course.
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 100,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})
sns.set_palette(PALETTE)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Environment ready.")
print(f"  numpy   {np.__version__}")
print(f"  pandas  {pd.__version__}")
print(f"  seaborn {sns.__version__}")
print(f"  seed    {RANDOM_SEED}")

---

# Part 1 — Theory and concepts

## 1.1 Long-form data

Almost every seaborn function expects **long-form** (tidy) data: one row per observation, one
column per variable. This is Week 1's definition and Week 6's `melt`.

```python
sns.boxplot(data=df, x="faculty", y="gpa")
```

`x` and `y` name **columns**, not arrays. seaborn does the grouping itself.

If your data is wide — one column per group — reshape it first:

```python
long = wide.melt(var_name="faculty", value_name="gpa")
```

This is the most common reason a seaborn call fails or produces something unexpected: the data
is in the wrong shape. Check with `.head()` before debugging the plotting call.

## 1.2 Axes-level and figure-level functions

seaborn has two kinds of function, and the difference decides whether a plot can be placed
inside a larger figure.

| Kind | Examples | Accepts `ax=` | Returns |
|---|---|:---:|---|
| **Axes-level** | `histplot`, `boxplot`, `violinplot`, `scatterplot`, `lineplot`, `heatmap`, `barplot`, `kdeplot` | yes | the `Axes` |
| **Figure-level** | `displot`, `catplot`, `relplot`, `pairplot`, `jointplot`, `lmplot` | **no** | a `FacetGrid` or similar |

**Axes-level functions compose.** They draw on an Axes you give them, exactly like the
`plot_category_summary` function from Week 7, so they can be panels of a grid.

**Figure-level functions create and own their own figure.** They cannot be placed inside a
subplot, and passing `ax=` raises. In exchange they handle faceting: `col=` and `row=` split
the data into a grid of panels automatically.

The naming is a reliable guide. The figure-level functions are the ones whose names end in
`plot` without a chart type in front — `displot` (distributions), `catplot` (categorical),
`relplot` (relational) — each of which is a wrapper that can produce several kinds of plot
through its `kind=` argument.

**Practical rule:** use axes-level functions by default, because they fit into the figures you
already know how to build. Reach for a figure-level function when you want faceting and the
plot is the whole output.

## 1.3 Distributions

| Function | Shows |
|---|---|
| `histplot` | Counts in bins. The bin count is still a choice (Week 7) |
| `kdeplot` | A smooth density estimate. The smoothing `bw_adjust` is also a choice |
| `ecdfplot` | The empirical cumulative distribution: no parameters at all |
| `rugplot` | A tick per observation, usually added beneath another plot |

`histplot(..., kde=True)` draws both. `stat="density"` rescales the histogram so it is
comparable with the KDE curve.

A KDE is a smoothed estimate, not the data. With a small sample it can invent structure that
is not there, and it always spills past the range of the observations — a density curve for
exam scores will show mass above 100. `ecdfplot` has no such parameter and is the honest
choice when the sample is small, at the cost of being harder to read.

## 1.4 Categorical plots, and what a box plot hides

A box plot draws the five-number summary from Week 1: the box spans $Q_1$ to $Q_3$, the line
inside is the median, and the whiskers reach to the furthest point within
$1.5 \times \mathrm{IQR}$ — Tukey's rule, which you implemented in Week 1's homework. Points
beyond that are drawn individually.

It is a compact and useful summary. It is also a **summary**, and the Week 1 lesson applies:
two distributions with identical five-number summaries can have entirely different shapes.

| Plot | Shows | Cost |
|---|---|---|
| `boxplot` | Five-number summary | Hides multimodality entirely |
| `violinplot` | A mirrored KDE, so the full shape | Inherits the KDE's smoothing assumptions |
| `stripplot` | Every observation, jittered | Unreadable above a few hundred points |
| `swarmplot` | Every observation, non-overlapping | Slow, and warns when points do not fit |
| `boxenplot` | More quantiles than a box plot | Less familiar to readers |

**The recommendation for this course:** draw a box plot with the points on top of it when the
sample is small enough, so the reader gets the summary and the raw data at once. Part 2
demonstrates why.

`barplot` deserves a separate warning. It aggregates with the **mean** by default and draws a
95% bootstrap confidence interval as the error bar. Many readers assume an error bar is a
standard deviation. State what it is, or set `errorbar=None` and say what you are showing.

## 1.5 Relational plots

```python
sns.scatterplot(data=df, x="attendance_pct", y="gpa", hue="faculty", size="credits")
sns.lineplot(data=df, x="date", y="visits")
```

The **semantic** arguments map a column onto a visual property:

| Argument | Maps a column to |
|---|---|
| `hue` | Colour |
| `size` | Marker size or line width |
| `style` | Marker shape or line dash |

Each one adds information and costs clarity. Two semantics on one chart is usually the limit.

`lineplot` has a behaviour worth knowing: when there are several `y` values for one `x`, it
**aggregates them** — mean and a 95% confidence band — rather than drawing every point. That
is useful for repeated measurements and surprising if you expected a raw line.

## 1.6 Matrix plots

```python
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0)
```

For a correlation matrix, three arguments matter and are often omitted:

- **`vmin=-1, vmax=1`** fixes the scale. Without it the colours are scaled to whatever range
  happens to be present, so two heatmaps cannot be compared and a weak correlation can look
  strong.
- **`center=0`** places the neutral colour at zero.
- **A diverging colormap** (`RdBu_r`, `coolwarm`) because correlation has a meaningful
  midpoint. A sequential map (`viridis`) implies that $-1$ is the "low" end of one scale,
  which misrepresents it: $-1$ and $+1$ are both strong.

A correlation matrix is symmetric, so half of it is redundant. Masking the upper triangle with
`np.triu` removes the duplication.

## 1.7 Pair plots

```python
sns.pairplot(df, vars=["gpa", "attendance_pct", "study_hours"], hue="faculty")
```

A grid of every variable against every other, with distributions on the diagonal. It is the
fastest way to look at a new dataset, and it is an **exploratory** tool: with $k$ variables it
draws $k^2$ panels, which is unreadable above about six and slow above about ten. Select the
columns you care about with `vars=`.

## 1.8 Themes

```python
sns.set_theme(style="whitegrid", palette="colorblind")
```

`set_theme` changes matplotlib's defaults globally, so it affects every subsequent plot
including pure matplotlib ones. Styles: `darkgrid`, `whitegrid`, `dark`, `white`, `ticks`.

This course sets its own `rcParams` and palette in the setup cell, so you do not need
`set_theme`. It is worth knowing because most seaborn examples begin with it.

---

---

# Part 2 - Code example

§2.4 is the centre of the session: two distributions with nearly identical box plots and
entirely different shapes.

## 2.1 The data

In [ ]:
def generate_cohort(n: int = 400, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Build a reproducible student cohort in long form.

    Returns
    -------
    pandas.DataFrame
        One row per student, with faculty, year level and four numeric measures.
    """
    generator = np.random.default_rng(seed)

    faculties = ["Engineering", "Business", "Science", "Arts"]
    faculty = generator.choice(faculties, size=n, p=[0.35, 0.30, 0.20, 0.15])
    ability = generator.normal(0, 1, size=n)

    frame = pd.DataFrame({
        "student_id": [f"S{2026000 + i}" for i in range(n)],
        "faculty": faculty,
        "year_level": generator.choice([1, 2, 3, 4], size=n, p=[0.30, 0.30, 0.25, 0.15]),
        "gpa": np.clip(2.6 + ability * 0.5, 0, 4).round(2),
        "attendance_pct": np.clip(78 + ability * 9 + generator.normal(0, 5, n),
                                  0, 100).round(1),
        "study_hours": np.clip(generator.normal(9, 3.2, n) + ability * 1.1, 0, None).round(1),
        "commute_min": generator.integers(5, 95, size=n),
    })
    return frame


cohort = generate_cohort()

print(f"{cohort.shape[0]} rows x {cohort.shape[1]} columns — long form, one row per student")
print(cohort.head())
print("\nThis is the shape seaborn wants: one row per observation, one column")
print("per variable. x= and y= name COLUMNS, not arrays.")

## 2.2 Axes-level and figure-level

The distinction that decides whether a plot can go inside a larger figure.

In [ ]:
# Axes-level functions accept ax= and compose into a grid, exactly like the
# Week 7 functions that took an ax argument.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

sns.histplot(data=cohort, x="gpa", bins=22, ax=axes[0], color=PALETTE[0])
axes[0].set_title("(a) histplot", loc="left")

sns.boxplot(data=cohort, x="faculty", y="gpa", ax=axes[1], hue="faculty",
            palette=PALETTE[:4], legend=False)
axes[1].set_title("(b) boxplot", loc="left")
axes[1].tick_params(axis="x", rotation=20)

sns.scatterplot(data=cohort, x="attendance_pct", y="gpa", ax=axes[2],
                s=18, alpha=0.5, color=PALETTE[2])
axes[2].set_title("(c) scatterplot", loc="left")

fig.suptitle("Axes-level functions accept ax= and compose",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Figure-level functions create and own their own figure. Passing ax= fails.
_, spare_ax = plt.subplots()
try:
    sns.displot(data=cohort, x="gpa", ax=spare_ax)
except TypeError as exc:
    print("sns.displot(..., ax=...) raises:")
    print(f"  TypeError: {str(exc)[:110]}...")
plt.close("all")

print("\nWhat they give in exchange is faceting: col= and row= split the data")
print("into a grid automatically, with shared scales.")

grid = sns.displot(data=cohort, x="gpa", col="faculty", col_wrap=2,
                   height=2.6, aspect=1.5, bins=18, color=PALETTE[0])
grid.figure.suptitle("displot with col='faculty'", y=1.03,
                     fontsize=12, fontweight="bold")
plt.show()

print("\nRULE: use axes-level functions by default, because they fit into the")
print("figures you already know how to build. Reach for a figure-level function")
print("when you want faceting and the plot is the whole output.")

## 2.3 Distributions

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))

sns.histplot(data=cohort, x="study_hours", bins=22, ax=axes[0], color=PALETTE[0])
axes[0].set_title("(a) histplot — counts in bins", loc="left")

sns.kdeplot(data=cohort, x="study_hours", ax=axes[1], color=PALETTE[1],
            fill=True, alpha=0.4)
axes[1].set_title("(b) kdeplot — smoothed density", loc="left")

sns.histplot(data=cohort, x="study_hours", bins=22, ax=axes[2], kde=True,
             stat="density", color=PALETTE[2])
axes[2].set_title("(c) both, stat='density'", loc="left")

sns.ecdfplot(data=cohort, x="study_hours", ax=axes[3], color=PALETTE[3])
axes[3].set_title("(d) ecdfplot — no parameters", loc="left")

plt.tight_layout()
plt.show()

print("Panel (c) needs stat='density' so the histogram and the curve share a")
print("vertical scale. Without it the KDE is flat against the counts.")

In [ ]:
# A KDE is an estimate with a parameter, and the parameter changes the story.
small = cohort["study_hours"].sample(30, random_state=RANDOM_SEED)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, adjust in zip(axes, [0.3, 1.0, 2.5]):
    sns.kdeplot(x=small, ax=ax, bw_adjust=adjust, fill=True,
                color=PALETTE[0], alpha=0.4)
    sns.rugplot(x=small, ax=ax, color=PALETTE[3], height=0.08)
    ax.set_title(f"bw_adjust = {adjust}", loc="left")
    ax.set_xlabel("study hours")

fig.suptitle("The same 30 observations, three smoothing settings",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("At 0.3 the curve invents several peaks that are just individual points.")
print("At 2.5 it flattens real structure away. The ticks underneath are the")
print("actual data — the only thing on this figure that is not an assumption.")
print("\nA KDE also spills past the range of the observations. Check the left")
print("edge: it shows density below zero hours, which is impossible.")

## 2.4 What a box plot hides

The Week 1 lesson, applied to distributions.

In [ ]:
def build_matched_groups(n: int = 300) -> pd.DataFrame:
    """Construct two samples whose quartiles agree to within 0.05.

    Both groups are built deterministically, by evaluating an inverse CDF at
    evenly spaced probabilities rather than by drawing random numbers. That
    removes sampling noise entirely, so the quartiles match exactly rather
    than approximately, and the result does not depend on a seed.

    Group A  a symmetric Beta(2, 2) scaled to [10, 90]: one central mode,
             bounded, with lighter tails than a normal.
    Group B  two normal clusters centred on A's quartiles. Because each
             cluster holds exactly half the data, the median of the lower
             cluster IS the 25th percentile of the mixture — so B's quartiles
             land on A's by construction.

    Returns
    -------
    pandas.DataFrame
        Long form: columns 'group' and 'score'.
    """
    probabilities = np.linspace(0.5 / n, 1 - 0.5 / n, n)

    low_bound, high_bound = 10.0, 90.0
    group_a = low_bound + (high_bound - low_bound) * stats.beta.ppf(
        probabilities, 2, 2
    )

    q1, q3 = np.percentile(group_a, [25, 75])
    offset = (q3 - q1) / 2          # places each cluster on a quartile of A

    half_probabilities = np.linspace(0.5 / (n // 2), 1 - 0.5 / (n // 2), n // 2)
    # Cluster width: wide enough for the tails to reach roughly A's range,
    # narrow enough for the two modes to separate visibly.
    spread = 0.75 * (q1 - low_bound) / stats.norm.ppf(1 - 0.5 / (n // 2))

    group_b = np.concatenate([
        stats.norm.ppf(half_probabilities, 50 - offset, spread),
        stats.norm.ppf(half_probabilities, 50 + offset, spread),
    ])

    return pd.DataFrame({
        "group": ["A (one mode)"] * len(group_a) + ["B (two modes)"] * len(group_b),
        "score": np.concatenate([group_a, group_b]),
    })


matched = build_matched_groups()

summary = matched.groupby("group")["score"].describe()[
    ["count", "min", "25%", "50%", "75%", "max"]].round(2)
summary["IQR"] = (summary["75%"] - summary["25%"]).round(2)

print("The five-number summary of each group:")
print(summary.to_string())

gaps = {
    "Q1": abs(summary["25%"].iloc[0] - summary["25%"].iloc[1]),
    "median": abs(summary["50%"].iloc[0] - summary["50%"].iloc[1]),
    "Q3": abs(summary["75%"].iloc[0] - summary["75%"].iloc[1]),
    "IQR": abs(summary["IQR"].iloc[0] - summary["IQR"].iloc[1]),
}
print("\nDifference between the two groups:")
for name, value in gaps.items():
    print(f"  {name:<8}{value:>8.2f}")

print("\nThe box of a box plot is drawn from Q1, the median and Q3. All three")
print("agree here to within a twentieth of a point, on a scale running from")
print("10 to 90.")
print("\nOn these numbers the two groups are the same.")

In [ ]:
# The box plot draws exactly those numbers, and nothing else.
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=matched, x="group", y="score", ax=ax,
            hue="group", palette=[PALETTE[0], PALETTE[1]], legend=False)
ax.set_title("Two groups, drawn as box plots", loc="left")
ax.set_xlabel("")
ax.set_ylabel("score")
plt.tight_layout()
plt.show()

print("Two boxes that are, to the eye, the same: same median line, same top,")
print("same bottom.")
print("\nThe whiskers differ a little, because the two groups reach slightly")
print("different extremes. Notice what that tells you: something about the two")
print("groups is not identical. It does not tell you WHAT, and nothing here")
print("suggests the answer is 'one of them has two separate clusters'.")
print("\nOn this evidence you would report that the groups are equivalent.")

In [ ]:
# Now show the same data four ways.
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

sns.boxplot(data=matched, x="group", y="score", ax=axes[0],
            hue="group", palette=[PALETTE[0], PALETTE[1]], legend=False)
axes[0].set_title("(a) boxplot", loc="left")

sns.violinplot(data=matched, x="group", y="score", ax=axes[1],
               hue="group", palette=[PALETTE[0], PALETTE[1]], legend=False)
axes[1].set_title("(b) violinplot", loc="left")

sns.stripplot(data=matched, x="group", y="score", ax=axes[2], size=2.5,
              alpha=0.45, hue="group", palette=[PALETTE[0], PALETTE[1]],
              legend=False, jitter=0.3)
axes[2].set_title("(c) stripplot", loc="left")

sns.histplot(data=matched, x="score", hue="group", ax=axes[3], bins=30,
             element="step", fill=False, palette=[PALETTE[0], PALETTE[1]])
axes[3].set_title("(d) histplot, overlaid", loc="left")
axes[3].set_xlabel("score")

for ax in axes[:3]:
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelsize=8)
axes[0].set_ylabel("score")

fig.suptitle("The same two groups. Only (a) says they are the same.",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Group B has two separate clusters with a clear gap between them.")
print("The box plot cannot show that, because the five-number summary does not")
print("contain it — the quartiles fall in the same places either way.")

# Quantify the gap, rather than asking the reader to take it on trust.
density = {
    name: stats.gaussian_kde(group["score"].to_numpy())
    for name, group in matched.groupby("group")
}
print("\nDensity at the median, relative to the density at the busiest point:")
for name, kde in density.items():
    grid = np.linspace(10, 90, 400)
    values = kde(grid)
    print(f"  {name:<16}{kde(50)[0] / values.max():.2f}")
print("\nGroup A is at its most crowded around the median. Group B is at barely")
print("half its peak there — the middle is where its students are NOT.")

print("\nThis is Anscombe's quartet again. A box plot is a summary statistic")
print("drawn as a picture, and summaries discard information.")

In [ ]:
# The practical recommendation: box plus points, when the sample allows it.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)

sns.boxplot(data=matched, x="group", y="score", ax=axes[0],
            hue="group", palette=[PALETTE[0], PALETTE[1]], legend=False)
axes[0].set_title("(a) Box plot alone", loc="left")

sns.boxplot(data=matched, x="group", y="score", ax=axes[1], width=0.45,
            showfliers=False, boxprops={"alpha": 0.5},
            hue="group", palette=[PALETTE[0], PALETTE[1]], legend=False)
sns.stripplot(data=matched, x="group", y="score", ax=axes[1], size=2,
              alpha=0.35, color="black", jitter=0.28)
axes[1].set_title("(b) Box plot with the observations on top", loc="left")

for ax in axes:
    ax.set_xlabel("")
axes[0].set_ylabel("score")

plt.tight_layout()
plt.show()

print("Panel (b) gives the reader the summary AND the shape. showfliers=False")
print("stops the outliers being drawn twice, once by each layer.")
print("\nUse this whenever the sample is small enough for the points to be")
print("legible — up to a few hundred per group.")

## 2.5 Categorical comparisons on the real cohort

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.boxplot(data=cohort, x="faculty", y="gpa", ax=axes[0], width=0.5,
            showfliers=False, boxprops={"alpha": 0.55},
            hue="faculty", palette=PALETTE[:4], legend=False)
sns.stripplot(data=cohort, x="faculty", y="gpa", ax=axes[0], size=2,
              alpha=0.3, color="black", jitter=0.25)
axes[0].set_title("(a) GPA by faculty: box + points", loc="left")

sns.violinplot(data=cohort, x="faculty", y="gpa", ax=axes[1],
               hue="faculty", palette=PALETTE[:4], legend=False, inner="quartile")
axes[1].set_title("(b) violinplot, inner='quartile'", loc="left")

sns.barplot(data=cohort, x="faculty", y="gpa", ax=axes[2],
            hue="faculty", palette=PALETTE[:4], legend=False, errorbar=("ci", 95))
axes[2].set_title("(c) barplot — mean with 95% CI", loc="left")
axes[2].set_ylim(0, 4)

for ax in axes:
    ax.tick_params(axis="x", rotation=20)
    ax.set_xlabel("")

plt.tight_layout()
plt.show()

print("Panel (c) needs a warning. sns.barplot aggregates with the MEAN and")
print("draws a 95% bootstrap confidence interval as the error bar — not a")
print("standard deviation, which is what most readers assume.")
print("\nEither state what the bar is in the caption, or pass errorbar=None")
print("and show the spread some other way.")

## 2.6 Heatmaps

In [ ]:
numeric = cohort[["gpa", "attendance_pct", "study_hours", "commute_min", "year_level"]]
correlation = numeric.corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# Left: the default. The colour scale is fitted to the data present.
sns.heatmap(correlation, annot=True, fmt=".2f", ax=axes[0], cmap="viridis")
axes[0].set_title("(a) Defaults: scaled to the data, sequential map", loc="left")

# Right: fixed scale, diverging map, upper triangle masked.
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, annot=True, fmt=".2f", ax=axes[1], mask=mask,
            cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            linewidths=0.5, cbar_kws={"label": "correlation"})
axes[1].set_title("(b) vmin/vmax fixed, diverging, half masked", loc="left")

plt.tight_layout()
plt.show()

print("Three things are wrong with panel (a):")
print("  - the colour scale is fitted to whatever range is present, so the")
print("    same colour means different things in different heatmaps")
print("  - a sequential map implies -1 is the 'low' end of one scale;")
print("    in fact -1 and +1 are both STRONG, and 0 is the middle")
print("  - the matrix is symmetric, so half the panel is duplication")

## 2.7 Pair plots

In [ ]:
pair = sns.pairplot(
    cohort,
    vars=["gpa", "attendance_pct", "study_hours"],
    hue="faculty",
    palette=PALETTE[:4],
    height=2.1,
    plot_kws={"s": 12, "alpha": 0.5, "edgecolor": "none"},
    diag_kind="kde",
)
pair.figure.suptitle("Pair plot of three measures, coloured by faculty",
                     y=1.02, fontsize=12, fontweight="bold")
plt.show()

print("Every variable against every other, with distributions on the diagonal.")
print("\nThis is an EXPLORATORY tool. With k variables it draws k^2 panels:")
print("  3 variables ->  9 panels   (readable)")
print("  6 variables -> 36 panels   (borderline)")
print(" 10 variables -> 100 panels  (useless, and slow)")
print("\nSelect columns with vars=. Do not hand a pair plot to a reader as a")
print("finished figure — use it to find what is worth drawing properly.")

In [ ]:
# A finished figure, combining what the session covered.
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

sns.boxplot(data=cohort, x="faculty", y="attendance_pct", ax=axes[0, 0],
            width=0.5, showfliers=False, boxprops={"alpha": 0.55},
            hue="faculty", palette=PALETTE[:4], legend=False)
sns.stripplot(data=cohort, x="faculty", y="attendance_pct", ax=axes[0, 0],
              size=1.8, alpha=0.3, color="black", jitter=0.25)
axes[0, 0].set_title("(a) Attendance by faculty", loc="left")
axes[0, 0].set_xlabel("")
axes[0, 0].tick_params(axis="x", rotation=20)

sns.histplot(data=cohort, x="gpa", hue="faculty", ax=axes[0, 1], bins=20,
             element="step", fill=False, palette=PALETTE[:4])
axes[0, 1].set_title("(b) GPA distribution by faculty", loc="left")

sns.scatterplot(data=cohort, x="attendance_pct", y="gpa", ax=axes[1, 0],
                hue="faculty", palette=PALETTE[:4], s=20, alpha=0.6,
                edgecolor="none")
axes[1, 0].set_title("(c) Attendance against GPA", loc="left")
axes[1, 0].legend(fontsize=7, frameon=False, loc="lower right")

mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, annot=True, fmt=".2f", ax=axes[1, 1], mask=mask,
            cmap="RdBu_r", vmin=-1, vmax=1, center=0, linewidths=0.5,
            annot_kws={"size": 8}, cbar=False)
axes[1, 1].set_title("(d) Correlations", loc="left")
axes[1, 1].tick_params(labelsize=7)

fig.suptitle("Student cohort: distribution, comparison, relationship, correlation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Four questions, four chart types, one figure. Every panel is an")
print("axes-level seaborn call placed on an Axes from plt.subplots.")

---

# Part 3 — Guided workshop (in class)

Four tasks. Run the helper cell from Week 7 if your checks report a missing title — it is
repeated below for convenience.

In [ ]:
def get_any_title(ax) -> str:
    """Return an Axes title from whichever of the three positions it was set in."""
    for location in ("center", "left", "right"):
        text = ax.get_title(loc=location)
        if text:
            return text
    return ""


print("helper ready")

### Task 1 — A distribution, three ways

**Problem.** On the three panels of `axes1`, draw the distribution of `commute_min`:

1. `axes1[0]` — a histogram with 20 bins.
2. `axes1[1]` — a KDE, filled.
3. `axes1[2]` — an ECDF.

Give each panel a title.

**Hints**

- `sns.histplot(data=cohort, x="commute_min", bins=20, ax=axes1[0])`.
- `sns.kdeplot(..., fill=True)`, `sns.ecdfplot(...)`.
- All three are axes-level, so all three take `ax=`.

**Expected output.** Three panels, followed by `Task 1 complete`.

In [ ]:
fig1, axes1 = plt.subplots(1, 3, figsize=(14, 3.6))

# TODO 1: histogram, KDE and ECDF of commute_min, one per panel, each titled.


plt.tight_layout()
plt.show()

In [ ]:
# Verification for Task 1.
problems = []
if len(axes1[0].patches) != 20:
    problems.append(f"axes1[0] should be a 20-bin histogram; "
                    f"found {len(axes1[0].patches)} patches")
if not axes1[1].lines and not axes1[1].collections:
    problems.append("axes1[1] has no KDE — use sns.kdeplot(..., ax=axes1[1])")
if not axes1[2].lines:
    problems.append("axes1[2] has no ECDF — use sns.ecdfplot(..., ax=axes1[2])")
for i in range(3):
    if not get_any_title(axes1[i]):
        problems.append(f"axes1[{i}] has no title")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 1 complete. Three views of the same variable.")

### Task 2 — Box plot with the observations on top

**Problem.** On `ax2`, draw `study_hours` by `faculty` as a box plot with the individual
points overlaid, following the recommendation in §2.4.

Requirements:

- the box plot must pass `showfliers=False`, so outliers are not drawn twice,
- a strip plot on top of it,
- a title and a y-axis label.

**Hints**

- Draw the box plot first, then the strip plot on the same `ax`.
- `sns.boxplot(..., showfliers=False, width=0.5, boxprops={"alpha": 0.55})`.
- `sns.stripplot(..., size=2, alpha=0.3, color="black", jitter=0.25)`.

**Expected output.** Four boxes with points over them, followed by `Task 2 complete`.

In [ ]:
fig2, ax2 = plt.subplots(figsize=(8, 4.5))

# TODO 2: box plot of study_hours by faculty, showfliers=False, points on top.


plt.tight_layout()
plt.show()

In [ ]:
# Verification for Task 2.
problems = []
if not ax2.patches:
    problems.append("no box plot found — use sns.boxplot(..., ax=ax2)")
if not ax2.collections:
    problems.append("no points found — add sns.stripplot(..., ax=ax2) on top")
elif len(ax2.collections) < 2:
    problems.append("only one layer of points; the strip plot should add "
                    "one collection per faculty")
if not get_any_title(ax2):
    problems.append("no title")
if not ax2.get_ylabel():
    problems.append("no y-axis label")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 2 complete. Summary and raw data in one panel.")

### Task 3 — A correlation heatmap done properly

**Problem.** On `ax3`, draw a heatmap of the correlations between `gpa`, `attendance_pct`,
`study_hours` and `commute_min`, with:

- the values annotated to 2 decimal places,
- a **diverging** colormap,
- the scale fixed to `vmin=-1`, `vmax=1`, centred on 0,
- a title.

**Hints**

- `cohort[[...]].corr()` gives the matrix.
- `sns.heatmap(matrix, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0, ax=ax3)`.

**Expected output.** A 4×4 annotated heatmap, followed by `Task 3 complete`.

In [ ]:
fig3, ax3 = plt.subplots(figsize=(6, 5))

# TODO 3: correlation heatmap with a fixed, centred, diverging scale.


plt.tight_layout()
plt.show()

In [ ]:
# Verification for Task 3.
problems = []
if not ax3.collections:
    problems.append("no heatmap found — use sns.heatmap(..., ax=ax3)")
else:
    mesh = ax3.collections[0]
    low, high = mesh.get_clim()
    if not (np.isclose(low, -1) and np.isclose(high, 1)):
        problems.append(f"the colour scale runs {low:.2f} to {high:.2f}; "
                        "fix it with vmin=-1, vmax=1 so heatmaps are comparable")
    if not ax3.texts:
        problems.append("no annotations — pass annot=True")
    elif len(ax3.texts) < 16:
        problems.append(f"expected 16 annotated cells for a 4x4 matrix; "
                        f"found {len(ax3.texts)}")
if not get_any_title(ax3):
    problems.append("no title")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 3 complete. Fixed scale, so this heatmap can be compared with another.")

### Task 4 — Choose the right plot

**Problem.** For each question below, name the seaborn function you would use, and store your
answers in the `answers` dictionary as lower-case function names without the `sns.` prefix.

| Key | Question |
|---|---|
| `"shape"` | What is the shape of the `gpa` distribution? |
| `"compare"` | How does `study_hours` differ between faculties, showing the full shape of each? |
| `"relate"` | Is `attendance_pct` related to `gpa`? |
| `"matrix"` | Which of the numeric variables move together? |

**Hints**

- Four different functions, all covered in Part 2.
- For `"compare"`, the requirement "showing the full shape" rules out the box plot.

**Expected output.** Four answers, followed by `Task 4 complete`.

In [ ]:
# TODO 4: fill in the four function names, lower case, no "sns." prefix.
answers = {
    "shape": None,
    "compare": None,
    "relate": None,
    "matrix": None,
}

for key, value in answers.items():
    print(f"  {key:<10} -> {value}")

In [ ]:
# Verification for Task 4.
acceptable = {
    "shape": {"histplot", "kdeplot", "displot", "ecdfplot"},
    "compare": {"violinplot", "stripplot", "swarmplot", "boxenplot"},
    "relate": {"scatterplot", "regplot", "relplot", "lmplot", "jointplot"},
    "matrix": {"heatmap", "clustermap", "pairplot"},
}

problems = []
for key, valid in acceptable.items():
    given = answers.get(key)
    if given is None:
        problems.append(f"{key!r} is still None")
    elif str(given).lower().replace("sns.", "") not in valid:
        problems.append(f"{key!r}: {given!r} is not one of {sorted(valid)}")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
    if any("compare" in p for p in problems):
        print("\n  For 'compare', note the requirement: the FULL SHAPE of each")
        print("  group. A box plot shows five numbers, not the shape — that is")
        print("  the whole point of section 2.4.")
else:
    print("Task 4 complete.")

---

# Part 4 — Homework

Run **Restart Kernel and Run All Cells** before submitting, then push to your
`puc-data-analysis` repository.

## Problem 1 — A distribution comparison that does not hide the shape

**Problem.** Write `plot_group_distributions(df, value_col, group_col, ax=None, show_points=True)`.

It draws the §2.4 recommendation: a box plot with the observations overlaid, so the reader
gets the summary and the shape at once.

**Requirements**

| Requirement | Detail |
|---|---|
| Return value | The `Axes` drawn on |
| `ax=None` | Create a new figure and axes; otherwise draw on the one given |
| Box plot | One box per group, drawn with `showfliers=False` |
| `show_points=True` | Overlay a strip plot of the observations |
| `show_points=False` | Draw the box plot **with** `showfliers=True`, so outliers are not lost |
| Labels | y-axis labelled with `value_col`, x-axis with `group_col`, and a non-empty title |
| Spines | Top and right hidden |

**Requirements on errors**

- Raise `KeyError` if either column is absent.
- Raise `TypeError` if `value_col` is not numeric.

Do not modify the input frame, and do not call `plt.show()`.

**Hints**

- Draw the box plot first, then the strip plot on the same `ax`.
- The `showfliers` value depends on `show_points`: hide them when the points are drawn (they
  would appear twice), show them when they are not (or the outliers vanish entirely).
- `sns.boxplot(data=df, x=group_col, y=value_col, ax=ax, ...)`.
- Seaborn sets its own axis labels from the column names; call `ax.set_ylabel(...)` **after**
  the seaborn calls or it will be overwritten.

In [ ]:
# Problem 1 — starter code.

def plot_group_distributions(df, value_col: str, group_col: str,
                             ax=None, show_points: bool = True):
    """Draw a box plot of value_col by group_col, optionally with the points on top.

    Parameters
    ----------
    df : pandas.DataFrame
        The data. Not modified.
    value_col : str
        Numeric column to summarise.
    group_col : str
        Column holding the groups.
    ax : matplotlib.axes.Axes or None
        Axes to draw on. A new figure is created when None.
    show_points : bool
        True overlays the observations and hides the box plot's outlier markers.
        False draws the box plot alone, with its outliers shown.

    Returns
    -------
    matplotlib.axes.Axes

    Raises
    ------
    KeyError
        If either column is absent.
    TypeError
        If value_col is not numeric.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete plot_group_distributions()")


demo_ax = plot_group_distributions(cohort, "study_hours", "faculty")
plt.tight_layout()
plt.show()

In [ ]:
# Problem 1 — validation. All assertions must pass.
import matplotlib

sample = pd.DataFrame({
    "team": ["A"] * 40 + ["B"] * 40,
    "score": np.concatenate([
        np.random.default_rng(1).normal(50, 10, 40),
        np.random.default_rng(2).normal(60, 10, 40),
    ]),
    "label": ["x"] * 80,
})

test_ax = plot_group_distributions(sample, "score", "team")

# --- return type ---
assert isinstance(test_ax, matplotlib.axes.Axes), (
    f"Must return an Axes, got {type(test_ax).__name__}"
)

# --- the box plot is there ---
assert test_ax.patches, "No box plot found — the boxes are patches on the Axes"

# --- with show_points=True the observations are overlaid ---
assert test_ax.collections, (
    "With show_points=True the observations must be drawn on top — "
    "sns.stripplot adds a collection per group"
)

# --- labels ---
assert test_ax.get_ylabel(), "The y-axis must be labelled"
assert "score" in test_ax.get_ylabel(), (
    f"The y-axis label should mention the value column; got {test_ax.get_ylabel()!r}. "
    "Note that seaborn sets its own labels — call set_ylabel AFTER the seaborn call."
)
assert test_ax.get_xlabel(), "The x-axis must be labelled"
assert get_any_title(test_ax), "The chart must have a title"

# --- spines ---
assert not test_ax.spines["top"].get_visible(), "Hide the top spine"
assert not test_ax.spines["right"].get_visible(), "Hide the right spine"

# --- the ax argument is honoured ---
_, spare = plt.subplots()
returned = plot_group_distributions(sample, "score", "team", ax=spare)
assert returned is spare, "When ax is given, draw on it and return that same Axes"

# --- show_points=False draws no strip layer ---
_, plain = plt.subplots()
plain_ax = plot_group_distributions(sample, "score", "team", ax=plain,
                                    show_points=False)
assert plain_ax.patches, "The box plot must still be drawn when show_points=False"
n_points_shown = sum(len(c.get_offsets()) for c in plain_ax.collections)
n_points_overlaid = sum(len(c.get_offsets()) for c in test_ax.collections)
assert n_points_overlaid > n_points_shown, (
    f"show_points=True should draw many more points than show_points=False. "
    f"Got {n_points_overlaid} and {n_points_shown}. With show_points=True, "
    "overlay a stripplot; with False, draw the box plot alone."
)

# --- the input is untouched ---
before = sample.copy()
_, throwaway = plt.subplots()
plot_group_distributions(sample, "score", "team", ax=throwaway)
assert sample.equals(before), "The function must not modify its input"

# --- errors ---
_, err_ax = plt.subplots()
for bad_value, bad_group in [("nope", "team"), ("score", "nope")]:
    try:
        plot_group_distributions(sample, bad_value, bad_group, ax=err_ax)
    except KeyError:
        pass
    else:
        raise AssertionError(f"A missing column ({bad_value}, {bad_group}) "
                             "must raise KeyError")

try:
    plot_group_distributions(sample, "label", "team", ax=err_ax)
except TypeError:
    pass
else:
    raise AssertionError("A non-numeric value_col must raise TypeError")

plt.close("all")
print("Problem 1: all assertions passed.")

---

## Problem 2 — A correlation heatmap with the defaults fixed

**Problem.** Write `correlation_heatmap(df, columns=None, ax=None, mask_upper=True)`.

**Requirements.** Return a tuple `(ax, corr)` where `corr` is the correlation matrix as a
`DataFrame`.

| Requirement | Detail |
|---|---|
| `columns=None` | Use every numeric column of `df` |
| `columns` given | Use exactly those columns, in that order |
| Colour scale | **Fixed** to `vmin=-1`, `vmax=1`, centred on 0 |
| Colormap | A diverging one |
| Annotation | Values shown to 2 decimal places |
| `mask_upper=True` | Hide the upper triangle **including** the diagonal |
| `mask_upper=False` | Show the whole matrix |
| Title | Non-empty |

**Requirements on errors**

- Raise `ValueError` if fewer than two numeric columns are available — a correlation matrix of
  one variable is meaningless.
- Raise `KeyError` if a name in `columns` is absent from `df`.

Do not modify the input frame.

**Hints**

- `df.select_dtypes(include="number").columns` gives the numeric columns. Note that this is
  the function that **excludes** booleans — the Week 4 distinction.
- `np.triu(np.ones_like(corr, dtype=bool))` masks the upper triangle *including* the diagonal.
  `np.triu(..., k=1)` would keep the diagonal, which is not what is asked for.
- Validate before computing, so the error names the real problem.
- The fixed scale is the whole point: without `vmin`/`vmax`, two heatmaps drawn from different
  data use different colour scales and cannot be compared.

In [ ]:
# Problem 2 — starter code.

def correlation_heatmap(df, columns=None, ax=None, mask_upper: bool = True):
    """Draw a correlation heatmap with a fixed, centred, diverging colour scale.

    Parameters
    ----------
    df : pandas.DataFrame
        The data. Not modified.
    columns : list of str or None
        Columns to correlate. None uses all numeric columns.
    ax : matplotlib.axes.Axes or None
        Axes to draw on. A new figure is created when None.
    mask_upper : bool
        Hide the upper triangle including the diagonal.

    Returns
    -------
    (matplotlib.axes.Axes, pandas.DataFrame)

    Raises
    ------
    ValueError
        If fewer than two numeric columns are available.
    KeyError
        If a named column is absent.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete correlation_heatmap()")


demo_ax, demo_corr = correlation_heatmap(
    cohort, columns=["gpa", "attendance_pct", "study_hours", "commute_min"]
)
plt.tight_layout()
plt.show()
print(demo_corr.round(3))

In [ ]:
# Problem 2 — validation. All assertions must pass.
import matplotlib

sample = pd.DataFrame({
    "a": [1.0, 2.0, 3.0, 4.0, 5.0],
    "b": [2.0, 4.0, 6.0, 8.0, 10.0],      # perfectly correlated with a
    "c": [5.0, 4.0, 3.0, 2.0, 1.0],       # perfectly anti-correlated with a
    "text": ["p", "q", "r", "s", "t"],
})

test_ax, corr = correlation_heatmap(sample)

# --- return shape ---
assert isinstance(test_ax, matplotlib.axes.Axes), "First return value must be an Axes"
assert isinstance(corr, pd.DataFrame), "Second return value must be a DataFrame"

# --- the text column is excluded, leaving a 3x3 matrix ---
assert corr.shape == (3, 3), (
    f"Only the numeric columns should be correlated, giving 3x3; got {corr.shape}"
)
assert list(corr.columns) == ["a", "b", "c"]

# --- the correlations are right ---
assert np.isclose(corr.loc["a", "b"], 1.0), "a and b are perfectly correlated"
assert np.isclose(corr.loc["a", "c"], -1.0), "a and c are perfectly anti-correlated"
assert np.isclose(corr.loc["a", "a"], 1.0)

# --- the colour scale is fixed, which is the point of the problem ---
assert test_ax.collections, "No heatmap was drawn"
low, high = test_ax.collections[0].get_clim()
assert np.isclose(low, -1) and np.isclose(high, 1), (
    f"The colour scale must be fixed to (-1, 1); got ({low:.2f}, {high:.2f}). "
    "Without this, two heatmaps use different scales and cannot be compared."
)

# --- annotation ---
assert test_ax.texts, "Values must be annotated — pass annot=True"

# --- masking hides the upper triangle INCLUDING the diagonal ---
# A 3x3 matrix has 3 cells in the strict lower triangle.
n_masked_texts = len(test_ax.texts)
assert n_masked_texts == 3, (
    f"With mask_upper=True a 3x3 matrix should annotate 3 cells "
    f"(the strict lower triangle); got {n_masked_texts}. "
    "np.triu(..., k=1) keeps the diagonal — use np.triu(...) without k."
)

_, full_axes = plt.subplots()
full_ax, _ = correlation_heatmap(sample, ax=full_axes, mask_upper=False)
assert len(full_ax.texts) == 9, (
    f"With mask_upper=False all 9 cells should be annotated; got {len(full_ax.texts)}"
)

# --- title ---
assert get_any_title(test_ax), "The heatmap needs a title"

# --- explicit columns are honoured, in order ---
_, sub_axes = plt.subplots()
sub_ax, sub_corr = correlation_heatmap(sample, columns=["c", "a"], ax=sub_axes)
assert list(sub_corr.columns) == ["c", "a"], (
    f"Columns must be used in the order given; got {list(sub_corr.columns)}"
)
assert sub_corr.shape == (2, 2)

# --- the ax argument is honoured ---
assert sub_ax is sub_axes, "When ax is given, draw on it and return that same Axes"

# --- errors ---
_, err_ax = plt.subplots()
try:
    correlation_heatmap(pd.DataFrame({"only": [1.0, 2.0, 3.0]}), ax=err_ax)
except ValueError:
    pass
else:
    raise AssertionError("Fewer than two numeric columns must raise ValueError")

try:
    correlation_heatmap(sample, columns=["a", "nope"], ax=err_ax)
except KeyError:
    pass
else:
    raise AssertionError("A missing column name must raise KeyError")

# --- the input is untouched ---
before = sample.copy()
_, throwaway = plt.subplots()
correlation_heatmap(sample, ax=throwaway)
assert sample.equals(before), "The function must not modify its input"

plt.close("all")
print("Problem 2: all assertions passed.")

---

## Written question

Answer in the Markdown cell below, in 120–200 words.

Section 2.4 showed two groups whose box plots were almost identical and whose underlying
distributions were not: one had a single mode, the other had two separated clusters.

1. Explain, in terms of what a box plot actually draws, why it could not show the difference.
2. Name the Week 1 demonstration this repeats, and state the general principle both share.
3. You are given a dataset of 50,000 observations in eight groups. A strip plot would be
   unreadable at that size. What would you draw instead, and why?

*Replace this text with your answer.*

**1.**

**2.**

**3.**

---

## Submission checklist

- [ ] All four workshop tasks print their `complete` message.
- [ ] Problem 1 prints `all assertions passed`.
- [ ] Problem 2 prints `all assertions passed`.
- [ ] The written question is answered in 120–200 words.
- [ ] **Restart Kernel and Run All Cells** completes with no errors.
- [ ] Committed and pushed to your `puc-data-analysis` repository.

## Looking ahead to Week 9

Week 9 is exploratory data analysis: putting Weeks 5 to 8 together into a systematic first
pass over an unfamiliar dataset. Correlation, covariance and outlier detection get their own
treatment, and the `five_number_summary` function you wrote in Week 1 returns.

Before the session, read McKinney §5.3 and re-read §7.2 on detecting and filtering outliers.

---

## References

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. §9.2.

Pfaffelhuber, P. *Python for Data Analysis* (course notes). §8.10.
Available at `https://pfaffelh.github.io/python_for_data/`.

Waskom, M. L. (2021). seaborn: statistical data visualization. *Journal of Open Source
Software*, 6(60), 3021.